In [1]:
import re

BRANDS = [
    "Samsung","Xiaomi","Infinix","Apple","OPPO","Oppo","vivo","Vivo","Realme","Tecno","Huawei","Nokia","Lenovo","Asus","OnePlus"
]

def parse_specs_from_title(title: str):
    t = title.lower()

    # RAM/ROM pola umum (contoh: "8/128", "8GB/128GB", "12gb 256gb", "8 128")
    ram = None
    rom = None
    for m in re.finditer(r'(?:(\d{1,2})\s*gb)?\s*[\/\s-]\s*(\d{2,4})\s*gb', t):
        g1, g2 = m.groups()
        # Heuristik: angka kecil = RAM, besar = ROM
        if g1 and g2:
            r1, r2 = int(g1), int(g2)
            if r1 <= 32 and r2 >= r1:  # RAM tipikal ≤ 32GB
                ram, rom = r1, r2
                break
    if ram is None or rom is None:
        # fallback pola terpisah
        m_ram = re.search(r'(\d{1,2})\s*gb', t)
        m_rom = re.search(r'(\d{2,4})\s*gb(?!\/)', t)
        if m_ram: ram = int(m_ram.group(1))
        if m_rom: 
            val = int(m_rom.group(1))
            if not ram or val >= 64: rom = val

    # Kamera utama, contoh: "50MP", "108 MP"
    cam = None
    m_cam = re.search(r'(\d{2,4})\s*mp\b', t)
    if m_cam:
        cam = int(m_cam.group(1))

    # Brand + Seri (brand = token pertama yang match BRANDS)
    brand = None
    for b in BRANDS:
        if b.lower() in t:
            brand = b.capitalize() if b.lower() != "oppo" else "OPPO"
            break

    series = title
    if brand:
        # hapus brand dari depan jika muncul
        series = re.sub(rf'^{brand}\s*', '', series, flags=re.I)

    return brand, series.strip(), ram, rom, cam


In [2]:
import time
import math
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from parsel import Selector

In [3]:
def launch_driver(headless=True):
    from selenium.webdriver.chrome.options import Options
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--start-maximized")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=id-ID")
    # User-Agent sederhana
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    return driver

In [4]:
def launch_driver(headless=True):
    from selenium.webdriver.chrome.options import Options
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--start-maximized")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=id-ID")
    # User-Agent sederhana
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    return driver

In [5]:
def scrape_shopee(max_items=1000, query="handphone"):
    driver = launch_driver(headless=True)
    # URL pencarian (urut relevansi). Kamu bisa tambah filter brand di kueri.
    url = f"https://shopee.co.id/search?keyword={query}"
    driver.get(url)
    time.sleep(5)

    data = []
    page = 0
    while len(data) < max_items and page < 50:
        # Scroll untuk render card
        for _ in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.0)

        sel = Selector(text=driver.page_source)
        cards = sel.css('div.shopee-search-item-result__item')  # fallback umum
        # Jika class berubah, gunakan [data-sqe="item"] sebagai alternatif:
        if not cards:
            cards = sel.css('[data-sqe="item"]')

        for c in cards:
            title = c.css('div._1NoI8_::text, div._1NoI8_ span::text, .yQmmFK::text').get()
            price_txt = c.css('span._1xk7ak::text, ._1xk7ak::text, ._1d9_77::text').get()
            if not title or not price_txt:
                continue

            digits = ''.join(ch for ch in price_txt if ch.isdigit())
            price_num = int(digits) if digits else None

            brand, series, ram, rom, cam = parse_specs_from_title(title)

            data.append({
                "source": "Shopee",
                "merk": brand,
                "seri": series,
                "judul_listing": title,
                "harga_idr": price_num,
                "ram_gb": ram,
                "rom_gb": rom,
                "kamera_mp": cam,
                "page": page+1
            })

        # Next page via param ?page=
        page += 1
        next_url = f"https://shopee.co.id/search?keyword={query}&page={page}"
        driver.get(next_url)
        time.sleep(4)

    driver.quit()
    import pandas as pd
    df = pd.DataFrame(data).drop_duplicates(subset=["judul_listing","harga_idr"])
    return df.head(max_items)


In [ ]:
# Jalankan tokopedia & shopee, target total ≥ 1000 entri
#df_tokopedia = scrape_tokopedia(max_items=3, query="handphone")
df_shopee    = scrape_shopee(max_items=3, query="handphone")

#df_all = (
#    pd.concat([df_tokopedia, df_shopee], ignore_index=True)
#      .dropna(subset=["harga_idr"])
#      .drop_duplicates(subset=["judul_listing","harga_idr"])
#      .reset_index(drop=True)
#)

print("Total rows:", len(df_shopee))
df_all.to_csv("[local file]", index=False)